# Unfriendly Seating Process — Line Model

This notebook summarizes and visualizes the results produced by the three
scripts of the line model:

1. **Calc_Probabilities**: builds the Padovan sequence, defines the closed-form
   (static) and series-based (sequential) probability functions, and computes
   the full probability matrix `probability_matrix` (rows = total seats,
   columns = number of occupied seats) using a recursive relation.
2. **Calc_ExpectedValues**: computes the expected number of occupied seats for
   both the static and sequential models, across a range of line lengths.
3. **Verify_D_1**: checks the D_1 stochastic dominance property (the static
   model's cumulative distribution dominates the sequential model's, for a
   fixed n), and stores the cumulative distributions for later plotting.

Below, each result is loaded and plotted for a few representative values of n.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "Source"))

from Calc_Probabilities import (
    padovan_sequence,
    probability_matrix,
    probability_matrix_list,
    possible_indices,
    X_stat,
    X_seq,
    N_stat_k,
    N_seq_k,
)
from Calc_ExpectedValues import (
    expected_seats_sequential,
    expected_seats_static,
)
from Verify_D_1 import accumulated_data

ModuleNotFoundError: No module named 'Calc_Probabilities'

## Seat-occupation probability: X_stat vs X_seq

For a fixed line length n, compares the probability that seat i is occupied,
computed with the closed-form Padovan formula (X_stat) against the series
expansion (X_seq).

In [ ]:
def compare_seat_probabilities(n):
    """Plot X_stat and X_seq for each seat position in a line of length n."""
    seat_indices = list(range(1, n + 1))
    stat_probs = [X_stat(n, i) for i in seat_indices]
    seq_probs = [X_seq(n, i) for i in seat_indices]

    plt.figure(figsize=(10, 6))
    plt.plot(seat_indices, stat_probs, color="blue", marker='o', label="Static model")
    plt.plot(seat_indices, seq_probs, color="red", marker='x', label="Sequential model")
    plt.xlabel("Seat position")
    plt.ylabel("Probability that the seat is occupied")
    plt.title(f"n = {n}")
    plt.legend()
    plt.grid(True)
    plt.show()


for n in [5, 10, 15, 20]:
    compare_seat_probabilities(n)

## Occupied-seat count distribution: N_stat vs N_seq

For a fixed line length n, compares the probability of having exactly k
occupied seats, computed with the closed-form formula (N_stat) against the
series expansion (N_seq).

In [ ]:
def compare_occupied_count_distributions(n_values):
    """Plot N_stat_k and N_seq_k over the valid k range, for each n in n_values."""
    for n in n_values:
        ks = list(possible_indices(n))
        stat_probs = [N_stat_k(n, k) for k in ks]
        seq_probs = [N_seq_k(n, k) for k in ks]

        plt.figure(figsize=(10, 6))
        plt.plot(ks, stat_probs, color="blue", marker='o', label="Static model")
        plt.plot(ks, seq_probs, color="red", marker='x', label="Sequential model")
        plt.xlabel("Number of occupied seats (k)")
        plt.ylabel("Probability")
        plt.title(f"n = {n}")
        plt.legend()
        plt.grid(True)
        plt.show()


compare_occupied_count_distributions([5, 10, 15, 20])

## Probability matrix

Heatmap of `probability_matrix`: rows are total seats (n), columns are number
of occupied seats (k). Each entry is the probability of exactly k occupied
seats in a line of n seats, computed via the sequential model's recursion.

In [ ]:
n_rows, n_cols = probability_matrix.shape

plt.figure(figsize=(8, 8))
plt.imshow(probability_matrix, cmap='viridis', interpolation='nearest')
plt.colorbar(label='Probability')
plt.title(f'Probability matrix ({n_rows} x {n_cols})')
plt.xlabel('Columns: number of occupied seats')
plt.ylabel('Rows: total number of seats')
plt.tight_layout()
plt.show()

## Expected number of occupied seats

Compares the expected number of occupied seats between the static and
sequential models, as the total number of seats grows.

In [ ]:
n_values_expected = list(range(0, len(probability_matrix), 5))

plt.figure(figsize=(10, 6))
plt.plot(n_values_expected, expected_seats_static, label="Static model", marker='o', color="blue")
plt.plot(n_values_expected, expected_seats_sequential, label="Sequential model", marker='x', color="red")
plt.xlabel("Total number of seats (n)")
plt.ylabel("Expected number of occupied seats")
plt.legend()
plt.grid(True)
plt.show()

## Cumulative distributions: static vs sequential

For each selected n, compares the cumulative distribution of occupied seats
between the two models. This is the data underlying the D_1 property check.

In [ ]:
for n in [1, 2, 3, 4, 5, 10, 20, 50, 100]:
    data = accumulated_data[n]

    plt.figure(figsize=(10, 6))
    plt.plot(data["ks"], data["cumulative_stat"], label="Static model", marker='o', color="blue")
    plt.plot(data["ks"], data["cumulative_seq"], label="Sequential model", marker='x', color="red")
    plt.xlabel("Maximum number of occupied seats")
    plt.ylabel("Cumulative probability")
    plt.title(f"n = {n}")
    plt.legend()
    plt.grid(True)
    plt.show()